# 분류 모델 학습 노트북 v0 (원본 파일) 
- 📌 **이 파일 = 원본( 기초 틀)**
- 전체 코드 수정시 이 원본 파일 수정.


K-MHaS 9개 라벨 **다중 라벨 분류** · KcELECTRA 파인튜닝

| 가져다 쓰는 것 (B 파일) | 이 노트북에서 하는 것 (A) |
| --- | --- |
| `data/kmhas.py` → `load_kmhas`, `LABEL_NAMES` | ① 토크나이징 + 학습용 Dataset 변환 |
| `models/metrics.py` → `evaluate` | ② KcELECTRA를 9개 라벨 다중 분류로 불러오기 |
| | ③ Trainer 설정 · 학습 · 체크포인트 저장 |
| | ④ 학습 중 점수 계산에 `evaluate()` 연결 |

**실행 환경**: 학과 서버 (`ss-team17`, RTX A5000 24GB) · VS Code Remote-SSH


## 0. 설정 (실험할 때 여기만 바꾸기)

In [3]:
# ===== 설정 (실험할 때 여기만 바꾸기) =====
import os

MODEL_NAME = "beomi/KcELECTRA-base"   # W5 B: "klue/roberta-base"로 바꿔 비교
MAX_LENGTH = 64         # B 기록: 토큰 길이 95% 지점 37 → 32의 배수로 올려 64
BATCH_SIZE = 32
LR = 2e-5
EPOCHS = 1              # v0: 1, v1(W4): 3~5
SEED = 42
SAMPLE_SIZE = 1000      # v0: 1000, v1: None (전체 train 78,305건)
THRESHOLD = 0.5         # W6 B가 validation으로 조정
USE_CLASS_WEIGHT = False  # W5 B 불균형 처리 실험 (True면 드문 라벨에 loss 가중치)

# --- 경로 (서버) ---
PROJECT_DIR = os.path.realpath(os.path.expanduser("~/project"))   # data/, models/ 폴더가 있는 곳
SAVE_DIR = os.path.join(PROJECT_DIR, "checkpoints", "v0")         # v1 할 때 "v1"로

# --- 거의 안 바꾸는 값 ---
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

print("PROJECT_DIR:", PROJECT_DIR)
print("SAVE_DIR:", SAVE_DIR)

PROJECT_DIR: /home/team17/project
SAVE_DIR: /home/team17/project/checkpoints/v0


## 1. 환경 준비

- 패키지 설치: **처음 한 번만** 실행하고, 끝나면 위쪽 **Restart**로 커널을 다시 시작한 뒤 다음 셀부터 실행
- 서버에 이미 설치돼 있으면 건너뛰어도 됨 (아래 확인 셀에서 torch·GPU가 보이면 OK)
- B 파일 경로 확인 + `__init__.py` 자동 생성

In [4]:
import os, sys

os.makedirs(SAVE_DIR, exist_ok=True)

# B 파일 import 준비
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
for pkg in ["data", "models"]:
    os.makedirs(pkg, exist_ok=True)
    open(os.path.join(pkg, "__init__.py"), "a").close()   # import되려면 필요

missing = [f for f in ["data/kmhas.py", "models/metrics.py"] if not os.path.exists(f)]
assert not missing, f"B 파일이 없어요: {missing} → VS Code 탐색기에서 {PROJECT_DIR} 아래에 만들기"
print("B 파일 확인 OK / 체크포인트 저장 위치:", SAVE_DIR)

B 파일 확인 OK / 체크포인트 저장 위치: /home/team17/project/checkpoints/v0


## 2. 데이터 불러오기 (B의 `load_kmhas`)

In [5]:
import random
import numpy as np
import torch
from transformers import set_seed

from data.kmhas import load_kmhas, LABEL_NAMES
from models.metrics import evaluate

set_seed(SEED)   # random · numpy · torch 시드를 한 번에 고정
NUM_LABELS = len(LABEL_NAMES)   # 9
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "-")
if device == "cpu":
    print("⚠️ GPU가 안 잡혀요. 커널이 서버 Python인지, nvidia-smi에 GPU가 보이는지 확인")

train_df, val_df, test_df = load_kmhas()   # B 기록: train 78305 / val 8776 / test 21939
del test_df   # test는 W9 최종 평가 때만. 실수로 쓰지 않도록 바로 지움

if SAMPLE_SIZE:
    train_df = train_df.sample(n=min(SAMPLE_SIZE, len(train_df)), random_state=SEED).reset_index(drop=True)
print(f"학습에 쓰는 데이터: train {len(train_df)} / val {len(val_df)}")
train_df.head()

device: cuda | GPU: NVIDIA RTX A5000
[train] 빈 문장 -0, 중복 -421, test와 겹침 -251 → 78305건
[최종] train 78305 / val 8776 / test 21939
학습에 쓰는 데이터: train 1000 / val 8776


,text,labels
0,안무하기 바쁘네 투애니원이나 원더걸스처럼 무대를 가지고 놀줄 알아야지,"[0, 0, 0, 0, 0, 0, 0, 0, 1]"
1,"""전라도 광주에서 총선 나와라 그럼되지않나? 그 다음 영창가고.""","[0, 0, 0, 0, 0, 0, 0, 0, 1]"
2,"""황교안, 신천지 종교로 인정""","[0, 0, 0, 0, 0, 0, 0, 0, 1]"
3,와 진짜 역겹다 지들은 뭐잘났다고 저런 댓글을 달까,"[0, 0, 0, 0, 0, 0, 0, 0, 1]"
4,화사가 본격 여자들만 좋아한다는 여자연예인 아님? ㅋㅋ,"[0, 0, 0, 0, 0, 0, 0, 0, 1]"


## 3. ① 토크나이징 + 학습용 Dataset 변환

- 문장 → `input_ids`, `attention_mask` (길이 `MAX_LENGTH`에서 자름)
- KcELECTRA는 바이트 단위 BPE 토크나이저라 토큰이 `'Ġì...'`처럼 깨진 글자로 보임 → 정상
- 패딩은 배치마다 가장 긴 문장에 맞춤 (`DataCollatorWithPadding`) → 짧은 댓글이 많아서 더 빠름
- 다중 라벨은 BCE loss를 쓰므로 정답 라벨을 **float**로 바꿔야 함 (`[0,0,1,1,0,...]` → `[0.,0.,1.,1.,0.,...]`)

In [6]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

# 저장소의 토크나이저를 그대로 씀. KcELECTRA는 바이트 단위 BPE라
# 토큰이 'Ġì...'처럼 깨진 글자로 보이는 게 정상 (한글을 바이트로 쪼갠 표기)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 토크나이저 점검: [UNK] 비율이 높으면 설정이 잘못된 것 (W5에 모델을 바꿀 때도 확인용)
_chk = tokenizer(train_df["text"][:200].tolist())["input_ids"]
_unk = sum(i == tokenizer.unk_token_id for ids in _chk for i in ids) / sum(len(ids) for ids in _chk)
print(f"토크나이저: {type(tokenizer).__name__} | 단어 수 {len(tokenizer)} | [UNK] 비율 {_unk:.1%}")
assert _unk < 0.05, "[UNK]가 너무 많아요 → 토크나이저 설정 확인"

def to_dataset(df):
    ds = Dataset.from_dict({
        "text": df["text"].tolist(),
        "labels": [[float(x) for x in l] for l in df["labels"]],
    })
    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LENGTH),
                batched=True, remove_columns=["text"])
    return ds

train_ds = to_dataset(train_df)
val_ds = to_dataset(val_df)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 확인: 첫 문장이 어떻게 바뀌었는지
ex = train_ds[0]
print(train_df["text"][0])
print(tokenizer.convert_ids_to_tokens(ex["input_ids"])[:20])
print("labels:", ex["labels"])

토크나이저: TokenizersBackend | 단어 수 30000 | [UNK] 비율 0.3%


Map: 100%|██████████| 8776/8776 [00:00<00:00, 57045.29 examples/s]

안무하기 바쁘네 투애니원이나 원더걸스처럼 무대를 가지고 놀줄 알아야지
['ĠìķĪë', '¬´', 'íķĺê¸°', 'Ġë°ĶìģĺëĦ¤', 'ĠíĪ¬', 'ìķł', 'ëĭĪ', 'ìĽĲ', 'ìĿ´ëĤĺ', 'ĠìĽĲ', 'ëįĶ', 'ê±¸', 'ìĬ¤', 'ì²ĺëŁ¼', 'Ġë¬´ëĮĢ', 'ë¥¼', 'Ġê°Ģì§Ģê³ł', 'ĠëĨĢ', 'ì¤Ħ', 'ĠìķĮìķĦìķ¼ì§Ģ']
labels: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]


## 4. ② KcELECTRA를 9개 라벨 다중 분류로 불러오기

- `problem_type="multi_label_classification"` → 출력 9개 각각에 sigmoid + BCE loss (한 문장에 라벨 여러 개 가능)
- `id2label`을 넣어두면 저장된 모델만 불러와도 라벨 이름이 따라옴 (W4 `predict()`, 데모에서 편함)
- `classifier ... newly initialized` 경고는 정상 (분류층은 새로 학습하는 부분)

In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    id2label={i: n for i, n in enumerate(LABEL_NAMES)},
    label2id={n: i for i, n in enumerate(LABEL_NAMES)},
)
print(model.config.problem_type, model.config.num_labels)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5062.45it/s]
[transformers] ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing 

multi_label_classification 9


## 5. ④ 학습 중 점수 계산에 B의 `evaluate()` 연결 + (선택) 클래스 가중치

- `compute_metrics`: 모델 출력(logit) → sigmoid 확률 → B의 `evaluate(y_true, y_prob, threshold)` → micro/macro F1
- 결과표 형식이 B 함수 하나로 통일돼서 W4 v1, W5 v2와 바로 비교 가능
- `USE_CLASS_WEIGHT=True`면 라벨별 `pos_weight = 음성 수 / 양성 수`로 드문 라벨(예: race 237건)을 더 크게 반영 (W5 B 실험용, v0는 False)

In [8]:
from transformers import Trainer

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    r = evaluate(labels, sigmoid(logits), threshold=THRESHOLD, show=False)
    out = {"micro_f1": r["micro_f1"], "macro_f1": r["macro_f1"]}
    out.update({f"f1_{k}": v for k, v in r["per_label_f1"].items()})
    return out

# 클래스 가중치 (USE_CLASS_WEIGHT=True일 때만 사용)
y_train = np.array(train_df["labels"].tolist())
pos = y_train.sum(axis=0)
pos_weight = torch.tensor((len(y_train) - pos) / np.maximum(pos, 1), dtype=torch.float)

class WeightedTrainer(Trainer):
    """BCE loss에 라벨별 pos_weight를 거는 Trainer"""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels.float())
        return (loss, outputs) if return_outputs else loss

print("train 라벨별 개수:", dict(zip(LABEL_NAMES, pos.astype(int))))
if USE_CLASS_WEIGHT:
    print("pos_weight:", dict(zip(LABEL_NAMES, pos_weight.numpy().round(1))))

train 라벨별 개수: {'origin': np.int64(96), 'physical': np.int64(84), 'politics': np.int64(104), 'profanity': np.int64(138), 'age': np.int64(60), 'gender': np.int64(78), 'race': np.int64(2), 'religion': np.int64(19), 'not_hate': np.int64(558)}


## 6. ③ Trainer 설정 · 학습 · 체크포인트 저장

- epoch마다 validation 평가 + 체크포인트를 `SAVE_DIR`(드라이브)에 저장
- 끝나면 **validation macro F1이 가장 좋은 체크포인트**를 자동으로 다시 불러옴 (불균형 데이터라 micro보다 macro 기준)
- 체크포인트는 최근 2개만 남김 (드라이브 용량 절약)

In [9]:
import math
from transformers import TrainingArguments

# warmup은 '단계 수(정수)'로 직접 계산 → transformers v4·v5 어느 버전이든 동작 (v5에서 warmup_ratio 인자가 없어짐)
total_steps = math.ceil(len(train_ds) / BATCH_SIZE) * EPOCHS
warmup_steps = math.ceil(total_steps * WARMUP_RATIO)
print(f"총 학습 step {total_steps} / warmup {warmup_steps}")

args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
)

TrainerClass = WeightedTrainer if USE_CLASS_WEIGHT else Trainer
trainer = TrainerClass(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(train_result)

총 학습 step 32 / warmup 4


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,F1 Origin,F1 Physical,F1 Politics,F1 Profanity,F1 Age,F1 Gender,F1 Race,F1 Religion,F1 Not Hate
1,0.586071,0.574354,0.521246,0.079324,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.713919


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.97it/s]


TrainOutput(global_step=32, training_loss=0.6250793226063251, metrics={'train_runtime': 7.0341, 'train_samples_per_second': 142.164, 'train_steps_per_second': 4.549, 'total_flos': 28224545603760.0, 'train_loss': 0.6250793226063251, 'epoch': 1.0})


## 7. validation 결과표 (B의 `evaluate`, 라벨별 리포트)

In [10]:
pred = trainer.predict(val_ds)
val_prob = sigmoid(pred.predictions)
result = evaluate(pred.label_ids, val_prob, threshold=THRESHOLD, show=True)

                precision    recall  f1-score   support

    origin(출신)       0.00      0.00      0.00       759
  physical(외모)       0.00      0.00      0.00       717
  politics(정치)       0.00      0.00      0.00       896
 profanity(욕설)       0.00      0.00      0.00      1291
       age(연령)       0.00      0.00      0.00       552
    gender(성별)       0.00      0.00      0.00       659
      race(인종)       0.00      0.00      0.00        24
  religion(종교)       0.00      0.00      0.00       140
not_hate(해당없음)       0.56      1.00      0.71      4887

     micro avg       0.56      0.49      0.52      9925
     macro avg       0.06      0.11      0.08      9925
  weighted avg       0.27      0.49      0.35      9925
   samples avg       0.55      0.55      0.55      9925

micro F1 0.5212 | macro F1 0.0793


## 8. 최종 모델 저장 + 저장 확인

- `SAVE_DIR/final`에 모델·토크나이저·설정값·결과를 함께 저장 → W4 `predict()`는 이 폴더만 불러오면 됨
- 저장한 폴더에서 다시 불러와 예시 문장을 예측해 봄 (서버 디스크에 제대로 저장됐는지 확인) (v0는 1,000건·1 epoch라 대부분 `not_hate`로 나와도 정상)

In [11]:
import json

FINAL_DIR = os.path.join(SAVE_DIR, "final")
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

config = dict(MODEL_NAME=MODEL_NAME, MAX_LENGTH=MAX_LENGTH, BATCH_SIZE=BATCH_SIZE, LR=LR,
              EPOCHS=EPOCHS, SEED=SEED, SAMPLE_SIZE=SAMPLE_SIZE, THRESHOLD=THRESHOLD,
              USE_CLASS_WEIGHT=USE_CLASS_WEIGHT, n_train=len(train_ds), n_val=len(val_ds))
with open(os.path.join(FINAL_DIR, "train_config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
with open(os.path.join(FINAL_DIR, "val_result.json"), "w", encoding="utf-8") as f:
    json.dump({k: (float(v) if not isinstance(v, dict) else {kk: float(vv) for kk, vv in v.items()})
               for k, v in result.items()}, f, ensure_ascii=False, indent=2)

print("저장된 체크포인트:", sorted(os.listdir(SAVE_DIR)))
print("final 폴더:", sorted(os.listdir(FINAL_DIR)))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.18it/s]

저장된 체크포인트: ['checkpoint-32', 'final']
final 폴더: ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'train_config.json', 'training_args.bin', 'val_result.json']


In [12]:
# 저장한 폴더에서 다시 불러와 예측 (저장이 제대로 됐는지 확인)
tok2 = AutoTokenizer.from_pretrained(FINAL_DIR)
model2 = AutoModelForSequenceClassification.from_pretrained(FINAL_DIR).to(device).eval()

samples = ["오늘 날씨 진짜 좋다", "이 정치인은 정말 최악이다", "나이 먹고 왜 저러냐"]
with torch.no_grad():
    enc = tok2(samples, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt").to(device)
    probs = torch.sigmoid(model2(**enc).logits).cpu().numpy()

id2label = model2.config.id2label
for s, p in zip(samples, probs):
    hits = [id2label[i] for i in np.where(p >= THRESHOLD)[0]]
    top = id2label[int(p.argmax())]
    print(f"{s}  →  {hits or '(없음)'}  | 최고 확률: {top} {p.max():.2f}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6023.03it/s]

오늘 날씨 진짜 좋다  →  ['not_hate']  | 최고 확률: not_hate 0.52
이 정치인은 정말 최악이다  →  ['not_hate']  | 최고 확률: not_hate 0.51
나이 먹고 왜 저러냐  →  ['not_hate']  | 최고 확률: not_hate 0.52


## 9. 기록 (노션 W3에 붙여넣기)

아래 셀 출력을 그대로 복사해서 노션에 붙이고, 7번 결과표는 캡쳐해서 같이 올리기.

**다음 (W4 v1)**: 설정 셀에서 `SAMPLE_SIZE = None`, `EPOCHS = 3`, `SAVE_DIR`의 `"v0"` → `"v1"`로 바꾸고 그대로 실행 (전체 데이터는 GPU에서 epoch당 몇 분 수준)

In [13]:
log = trainer.state.log_history
train_losses = [x["loss"] for x in log if "loss" in x]
print(f"""- 설정: {MODEL_NAME} / MAX_LENGTH {MAX_LENGTH} / batch {BATCH_SIZE} / lr {LR} / epoch {EPOCHS} / sample {SAMPLE_SIZE} / threshold {THRESHOLD} / class_weight {USE_CLASS_WEIGHT}
- 데이터: train {len(train_ds)} / val {len(val_ds)}
- 학습 시간: {train_result.metrics['train_runtime']:.0f}초 / train loss {(f"{train_losses[0]:.4f} → {train_losses[-1]:.4f}" if train_losses else f"{train_result.training_loss:.4f}")}
- validation: micro F1 {result['micro_f1']:.4f} | macro F1 {result['macro_f1']:.4f}
- 체크포인트 저장: {FINAL_DIR}""")

- 설정: beomi/KcELECTRA-base / MAX_LENGTH 64 / batch 32 / lr 2e-05 / epoch 1 / sample 1000 / threshold 0.5 / class_weight False
- 데이터: train 1000 / val 8776
- 학습 시간: 7초 / train loss 0.6730 → 0.5861
- validation: micro F1 0.5212 | macro F1 0.0793
- 체크포인트 저장: /home/team17/project/checkpoints/v0/final
